# Indian Startup Funding — Cleaning Pipeline

**Notebook 02 of 4** | Goal: Transform raw, messy data into a single analysis-ready dataset.

## What This Notebook Does
Takes the 4 raw datasets we explored in Notebook 01 and runs them through a systematic cleaning pipeline — fixing every issue we documented, making deliberate decisions at each step, and logging every choice.

## The Cleaning Pipeline (in order)
1. **Load raw data** — reload all sources fresh
2. **Fix column names** — standardize to snake_case across all sources
3. **Handle missing values** — real nulls + hidden nulls (em-dashes)
4. **Parse dates** — handle DD/MM/YYYY and DD-MM-YYYY formats
5. **Clean amounts** — strip commas, detect currency, normalize to USD
6. **Normalize company names** — Unicode, whitespace, legal suffixes
7. **Map funding stages** — 30+ raw labels → 8 canonical stages
8. **Consolidate sectors** — 50+ raw tags → 13 canonical sectors
9. **Reshape Dataset 4** — wide to long format using pd.melt()
10. **Merge sources** — combine Dataset 1 + Dataset 4 into master
11. **Fuzzy deduplication** — catch name variants of same company
12. **Final validation** — assert data quality before export
13. **Export** — save to funding_clean.csv + SQLite database

## Methodology Log
Every non-obvious decision made in this notebook is flagged with:
> 📋 **METHODOLOGY NOTE:** [decision + reasoning]

These notes become the methodology log deliverable.

In [1]:
# ─── Core libraries ───────────────────────────────────────
import pandas as pd
import numpy as np
import re
import os
import unicodedata
from rapidfuzz import fuzz, process

# ─── Display settings ─────────────────────────────────────
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ─── Suppress non-critical warnings ───────────────────────
import warnings
warnings.filterwarnings('ignore')

# ─── Path configuration ───────────────────────────────────
RAW_DIR       = '../data/raw/'
INTERIM_DIR   = '../data/interim/'
PROCESSED_DIR = '../data/processed/'

# Verify all directories exist
for path in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR]:
    if os.path.exists(path):
        print(f"✓ Found: {path}")
    else:
        print(f"✗ Missing: {path}")

print("\nAll libraries loaded successfully.")

✓ Found: ../data/raw/
✓ Found: ../data/interim/
✓ Found: ../data/processed/

All libraries loaded successfully.


---

## Step 1 — Load Raw Data

Reloading all sources fresh into this notebook. We never modify the raw files — every transformation produces a new object in memory or saves to `data/interim/`.

In [2]:
# ─── Load all raw sources ─────────────────────────────────

# Dataset 1 — primary source (2015-2020)
df1 = pd.read_csv(f'{RAW_DIR}startup_funding_raw_1.csv')
print(f"✓ Dataset 1 loaded: {df1.shape}")

# Dataset 2 — enrichment only (no dates)
df2 = pd.read_csv(f'{RAW_DIR}startup_funding_raw_2.csv')
print(f"✓ Dataset 2 loaded: {df2.shape}")

# Dataset 4 — Crunchbase global (filter India next)
df4 = pd.read_csv(f'{RAW_DIR}startup_funding_raw_4.csv',
                  encoding='latin-1',
                  low_memory=False)
print(f"✓ Dataset 4 loaded: {df4.shape}")

# Filter Dataset 4 for India immediately
df4 = df4[df4['country_code'] == 'IND'].copy()
df4.columns = df4.columns.str.strip()
print(f"✓ Dataset 4 filtered to India: {df4.shape}")

# Dataset 3 is excluded — documented in Notebook 01
print(f"\n📋 METHODOLOGY NOTE: Dataset 3 excluded — identified as")
print(f"   synthetic/fabricated during exploration (Notebook 01).")
print(f"\nSources ready for cleaning.")

✓ Dataset 1 loaded: (3044, 10)
✓ Dataset 2 loaded: (526, 6)
✓ Dataset 4 loaded: (54294, 39)
✓ Dataset 4 filtered to India: (849, 39)

📋 METHODOLOGY NOTE: Dataset 3 excluded — identified as
   synthetic/fabricated during exploration (Notebook 01).

Sources ready for cleaning.


---

## Step 2 — Fix Column Names

Raw column names are inconsistent across all sources:
- Spaces inside names (`City  Location`, `Amount in USD`)
- Typos (`InvestmentnType`)
- Mixed conventions (`Startup Name` vs `startup` vs `Company Name`)
- Format info baked into names (`Date dd/mm/yyyy`)

**Fix:** Rename all columns to clean `snake_case` — lowercase, words separated by underscores, no special characters.

This is always the first transformation — clean column names make every subsequent operation cleaner.

In [3]:
# ─── Standardize column names ─────────────────────────────

# Dataset 1 — rename all 10 columns
df1.columns = [
    'sr_no',
    'date',
    'startup_name',
    'industry',
    'sub_vertical',
    'city',
    'investors',
    'stage',
    'amount_usd',
    'remarks'
]

# Dataset 2 — rename all 6 columns
df2.columns = [
    'startup_name',
    'industry',
    'stage',
    'amount',
    'location',
    'about'
]

# Dataset 4 — select only useful columns + rename
df4 = df4[[
    'name', 'category_list', 'market', 'status',
    'city', 'funding_rounds', 'founded_year',
    'first_funding_at', 'last_funding_at',
    'seed', 'venture', 'angel', 'grant',
    'private_equity', 'round_A', 'round_B',
    'round_C', 'round_D', 'round_E', 'round_F',
    'funding_total_usd'
]].copy()

df4.rename(columns={
    'name': 'startup_name',
    'category_list': 'industry',
    'market': 'market_tag',
    'first_funding_at': 'first_funding_date',
    'last_funding_at': 'last_funding_date'
}, inplace=True)

# Verify
print("Dataset 1 columns:", list(df1.columns))
print("\nDataset 2 columns:", list(df2.columns))
print("\nDataset 4 columns:", list(df4.columns))

Dataset 1 columns: ['sr_no', 'date', 'startup_name', 'industry', 'sub_vertical', 'city', 'investors', 'stage', 'amount_usd', 'remarks']

Dataset 2 columns: ['startup_name', 'industry', 'stage', 'amount', 'location', 'about']

Dataset 4 columns: ['startup_name', 'industry', 'market_tag', 'status', 'city', 'funding_rounds', 'founded_year', 'first_funding_date', 'last_funding_date', 'seed', 'venture', 'angel', 'grant', 'private_equity', 'round_A', 'round_B', 'round_C', 'round_D', 'round_E', 'round_F', 'funding_total_usd']


---

## Step 3 — Drop Useless Columns

Some columns add no analytical value:
- `sr_no` — just a row number, pandas has its own index
- `remarks` — 86.2% empty, no consistent content
- `sub_vertical` — 30.7% empty, too granular for our analysis
- `about` (Dataset 2) — useful for context but not for analysis

📋 **METHODOLOGY NOTE:** Columns dropped when missing > 50% of values
AND the column doesn't contribute to any analytical question we're
answering. Sub_vertical is borderline (30.7% missing) but dropped
because our sector consolidation in Step 7 makes it redundant.

In [4]:
# ─── Drop useless columns ─────────────────────────────────

# Dataset 1 — drop sr_no, remarks, sub_vertical
df1.drop(columns=['sr_no', 'remarks', 'sub_vertical'], inplace=True)
print(f"Dataset 1: {df1.shape} — dropped sr_no, remarks, sub_vertical")

# Dataset 2 — drop about (enrichment text, not analytical)
df2.drop(columns=['about'], inplace=True)
print(f"Dataset 2: {df2.shape} — dropped about")

# Dataset 4 — no columns to drop (we already selected only useful ones)
print(f"Dataset 4: {df4.shape} — no drops needed")

# Show current columns
print("\nDataset 1 columns now:", list(df1.columns))
print("Dataset 2 columns now:", list(df2.columns))

Dataset 1: (3044, 7) — dropped sr_no, remarks, sub_vertical
Dataset 2: (526, 5) — dropped about
Dataset 4: (849, 21) — no drops needed

Dataset 1 columns now: ['date', 'startup_name', 'industry', 'city', 'investors', 'stage', 'amount_usd']
Dataset 2 columns now: ['startup_name', 'industry', 'stage', 'amount', 'location']


---

## Step 4 — Handle Missing Values

Two types of missing values to handle:
- **Real nulls** — pandas already sees these as NaN
- **Hidden nulls** — em-dashes `—`, hyphens `-`, `N/A`, `NA`, blank strings that pandas doesn't recognize as missing

We handle hidden nulls first, then decide what to do with real nulls per column.

📋 **METHODOLOGY NOTE:** Rows with missing `amount_usd` are kept — they
still represent real funding events and contribute to round counts,
geographic analysis, and stage analysis. They're only excluded from
amount-based calculations.

In [5]:
# ─── Handle missing values ────────────────────────────────

# Define all strings that mean "missing" across our datasets
MISSING_MARKERS = ['—', '-', 'N/A', 'NA', 'n/a', 'na', 
                   'none', 'None', 'NULL', 'null', '', ' ']

# Replace all missing markers with proper NaN in all datasets
for df in [df1, df2, df4]:
    df.replace(MISSING_MARKERS, np.nan, inplace=True)

# Verify — check Dataset 2 amount column specifically
print("Dataset 2 — Amount column after replacing hidden nulls:")
missing_now = df2['amount'].isnull().sum()
print(f"  Null count now: {missing_now} ({missing_now/len(df2)*100:.1f}%)")

# Full missing value report for Dataset 1
print("\nDataset 1 — Missing values after cleanup:")
for col in df1.columns:
    n = df1[col].isnull().sum()
    pct = n/len(df1)*100
    print(f"  {col:<20} {n:>4} missing  ({pct:.1f}%)")

Dataset 2 — Amount column after replacing hidden nulls:
  Null count now: 148 (28.1%)

Dataset 1 — Missing values after cleanup:
  date                    0 missing  (0.0%)
  startup_name            0 missing  (0.0%)
  industry              171 missing  (5.6%)
  city                  180 missing  (5.9%)
  investors              24 missing  (0.8%)
  stage                   4 missing  (0.1%)
  amount_usd            960 missing  (31.5%)


---

## Step 5 — Parse Dates

Dataset 1 dates are stored as strings in `DD/MM/YYYY` format.
Pandas needs proper datetime objects to do any time-based analysis.

Two things we do here:
1. Parse the date string into a real datetime object
2. Extract useful derived columns: `year`, `quarter`, `month`

📋 **METHODOLOGY NOTE:** Using `dayfirst=True` because column header
explicitly states `dd/mm/yyyy` format. Using `errors='coerce'` so
any unparseable dates become NaT (Not a Time) rather than crashing
the pipeline.

In [6]:
# ─── Parse dates ──────────────────────────────────────────

# Dataset 1 — DD/MM/YYYY format
df1['date'] = pd.to_datetime(df1['date'], 
                              dayfirst=True, 
                              errors='coerce')

# Check how many failed to parse
failed = df1['date'].isnull().sum()
print(f"Dataset 1 — Dates parsed successfully: {len(df1) - failed}/{len(df1)}")
print(f"Failed to parse: {failed}")

# Extract derived time columns
df1['year']    = df1['date'].dt.year
df1['quarter'] = df1['date'].dt.quarter
df1['month']   = df1['date'].dt.month

# Add funding era — categorical label for each market cycle
def assign_era(year):
    if pd.isna(year):
        return np.nan
    elif year <= 2016:
        return '2014-2016 Consumer Boom'
    elif year <= 2019:
        return '2017-2019 Rationalization'
    elif year <= 2021:
        return '2020-2021 ZIRP Mania'
    elif year <= 2023:
        return '2022-2023 Funding Winter'
    else:
        return '2024-2025 AI Recovery'

df1['funding_era'] = df1['year'].apply(assign_era)

# Verify
print("\nDate range in Dataset 1:")
print(f"  Earliest: {df1['date'].min()}")
print(f"  Latest:   {df1['date'].max()}")
print(f"\nFunding era distribution:")
print(df1['funding_era'].value_counts())

Dataset 1 — Dates parsed successfully: 3036/3044
Failed to parse: 8

Date range in Dataset 1:
  Earliest: 2015-01-02 00:00:00
  Latest:   2020-01-13 00:00:00

Funding era distribution:
funding_era
2014-2016 Consumer Boom      1922
2017-2019 Rationalization    1107
2020-2021 ZIRP Mania            7
Name: count, dtype: int64



---

## Step 6 — Clean Amount Column (Dataset 1)

The `amount_usd` column in Dataset 1 has two problems:
1. Stored as string — can't do math
2. Indian comma format — `20,00,00,000` means 20 crore, not 20 billion

**Fix:** Strip all commas and convert to integer.

📋 **METHODOLOGY NOTE:** The column is labelled "Amount in USD" in the
original source. We trust this label — amounts are treated as USD.
Rows with missing amounts (960 rows, 31.5%) are retained with NaN
and excluded only from amount-based calculations.

In [7]:
# ─── Clean amount column ──────────────────────────────────

# Step 1 — strip commas from amount_usd
df1['amount_usd'] = df1['amount_usd'].astype(str).str.replace(',', '', regex=False)

# Step 2 — convert to numeric (non-numeric becomes NaN)
df1['amount_usd'] = pd.to_numeric(df1['amount_usd'], errors='coerce')

# Step 3 — verify
total = len(df1)
has_amount = df1['amount_usd'].notna().sum()
missing_amt = df1['amount_usd'].isna().sum()

print("Amount column after cleaning:")
print(f"  Rows with valid amount:   {has_amount} ({has_amount/total*100:.1f}%)")
print(f"  Rows with missing amount: {missing_amt} ({missing_amt/total*100:.1f}%)")

print("\nAmount statistics (USD):")
print(f"  Min:    ${df1['amount_usd'].min():>15,.0f}")
print(f"  Median: ${df1['amount_usd'].median():>15,.0f}")
print(f"  Mean:   ${df1['amount_usd'].mean():>15,.0f}")
print(f"  Max:    ${df1['amount_usd'].max():>15,.0f}")

# Check for suspiciously small amounts (likely data errors)
tiny = df1[df1['amount_usd'] < 1000].dropna(subset=['amount_usd'])
print(f"\nSuspiciously small amounts (< $1,000): {len(tiny)} rows")
if len(tiny) > 0:
    print(tiny[['startup_name', 'stage', 'amount_usd']].head(5))

Amount column after cleaning:
  Rows with valid amount:   2065 (67.8%)
  Rows with missing amount: 979 (32.2%)

Amount statistics (USD):
  Min:    $         16,000
  Median: $      1,700,000
  Mean:   $     18,429,897
  Max:    $  3,900,000,000

Suspiciously small amounts (< $1,000): 0 rows


---

## Step 7 — Normalize Company Names

Before we can deduplicate companies across sources, we need to
standardize how names are written. The same company can appear as:
- `BYJU'S` vs `Byju's` vs `Byjus` (Unicode + case)
- `Flipkart Pvt Ltd` vs `Flipkart` (legal suffix)
- `  Ola  ` vs `Ola` (whitespace)
- `OlaCabs` vs `Ola Cabs` (spacing within name)

We create a new column `startup_name_clean` — keeping the original
untouched for reference.

📋 **METHODOLOGY NOTE:** We normalize to lowercase for matching
purposes only. The display name in the final dataset retains
original casing from Dataset 1 (our primary source).

In [8]:
# ─── Normalize company names ──────────────────────────────

def clean_company_name(name):
    if pd.isna(name):
        return np.nan
    
    # Step 1 — normalize Unicode (curly apostrophes → straight)
    name = unicodedata.normalize('NFKD', str(name))
    name = name.encode('ascii', 'ignore').decode('ascii')
    
    # Step 2 — lowercase
    name = name.lower()
    
    # Step 3 — remove legal suffixes
    legal_suffixes = [
        r'\bprivate limited\b', r'\bpvt\.?\s*ltd\.?\b',
        r'\bltd\.?\b', r'\binc\.?\b', r'\bcorp\.?\b',
        r'\bllp\b', r'\bllc\b', r'\bco\.?\b'
    ]
    for suffix in legal_suffixes:
        name = re.sub(suffix, '', name, flags=re.IGNORECASE)
    
    # Step 4 — remove URLs if present
    name = re.sub(r'http\S+|www\.\S+', '', name)
    
    # Step 5 — remove special characters except spaces
    name = re.sub(r'[^\w\s]', '', name)
    
    # Step 6 — collapse multiple spaces + strip
    name = re.sub(r'\s+', ' ', name).strip()
    
    return name

# Apply to all three datasets
df1['startup_name_clean'] = df1['startup_name'].apply(clean_company_name)
df2['startup_name_clean'] = df2['startup_name'].apply(clean_company_name)
df4['startup_name_clean'] = df4['startup_name'].apply(clean_company_name)

# Show before/after examples
print("Company name normalization — before vs after:")
print("=" * 60)
examples = df1[['startup_name', 'startup_name_clean']].drop_duplicates().head(15)
print(examples.to_string(index=False))

Company name normalization — before vs after:
                startup_name startup_name_clean
                      BYJU’S              byjus
                      Shuttl             shuttl
                   Mamaearth          mamaearth
https://www.wealthbucket.in/                   
                      Fashor             fashor
                       Pando              pando
                      Zomato             zomato
                      Ecozen             ecozen
                    CarDekho           cardekho
                Dhruva Space       dhruva space
                      Rivigo             rivigo
                  Healthians         healthians
                     Licious            licious
                      InCred             incred
                       Trell              trell


In [9]:
# ─── Fix empty names after cleaning ──────────────────────

# Find rows where cleaning produced empty string
empty_clean = df1[df1['startup_name_clean'].str.strip() == '']
print(f"Rows with empty clean name: {len(empty_clean)}")
print(empty_clean[['startup_name', 'startup_name_clean']])

# Fix — extract domain name from URL as company name
def fix_url_name(row):
    if str(row['startup_name_clean']).strip() == '':
        # Extract domain from original URL
        url = str(row['startup_name'])
        domain = re.sub(r'https?://(www\.)?', '', url)
        domain = domain.split('/')[0]        # remove path
        domain = domain.split('.')[0]        # take just the name part
        return domain.lower()
    return row['startup_name_clean']

df1['startup_name_clean'] = df1.apply(fix_url_name, axis=1)

# Verify fix
print("\nAfter fix:")
print(df1[df1['startup_name'].str.contains('http', na=False)][
    ['startup_name', 'startup_name_clean']].head())

Rows with empty clean name: 1
                   startup_name startup_name_clean
3  https://www.wealthbucket.in/                   

After fix:
                   startup_name startup_name_clean
3  https://www.wealthbucket.in/       wealthbucket


In [10]:
# ─── Verify deduplication improvement ─────────────────────

before = df1['startup_name'].nunique()
after = df1['startup_name_clean'].nunique()

print(f"Unique raw names:     {before}")
print(f"Unique cleaned names: {after}")
print(f"→ Collapsed {before - after} duplicates through normalization alone")

# Show examples of what got collapsed
# (multiple raw names mapping to same clean name)
grouped = df1.groupby('startup_name_clean')['startup_name'].unique()
collapsed = grouped[grouped.apply(len) > 1]

print(f"\nCompanies with multiple spellings collapsed: {len(collapsed)}")
print("\nExamples of collapsed variants:")
for clean_name, raw_names in collapsed.head(10).items():
    print(f"\n  {clean_name!r} ← {list(raw_names)}")

Unique raw names:     2459
Unique cleaned names: 2343
→ Collapsed 116 duplicates through normalization alone

Companies with multiple spellings collapsed: 109

Examples of collapsed variants:

  'aisle' ← ['Aisle', 'Aisle.co']

  'applop' ← ['AppLop', 'Applop']

  'appsdaily' ← ['AppsDaily', 'Appsdaily']

  'aye finance' ← ['Aye Finance', 'AYE Finance']

  'babychakra' ← ['BabyChakra', 'Babychakra']

  'betaout' ← ['BetaOut', 'Betaout']

  'betterplace' ← ['Betterplace', 'BetterPlace']

  'blackbuck' ← ['BlackBuck', 'Blackbuck']

  'byjus' ← ['BYJU’S', '"BYJU\\\\\'S"']

  'byjuxe2x80x99s' ← ['Byju\\\\xe2\\\\x80\\\\x99s', 'BYJU\\\\xe2\\\\x80\\\\x99s']


In [11]:
# ─── Check Dataset 4 name cleaning ────────────────────────

print("Dataset 4 — sample of cleaned names:")
sample = df4[['startup_name', 'startup_name_clean']].head(15)
print(sample.to_string(index=False))

print(f"\nDataset 4: {df4['startup_name'].nunique()} → {df4['startup_name_clean'].nunique()} unique")

Dataset 4 — sample of cleaned names:
         startup_name   startup_name_clean
               1CLICK               1click
           21Diamonds           21diamonds
        24x7 Learning        24x7 learning
                3DSoC                3dsoc
            91Mobiles            91mobiles
           99Presents           99presents
              99tests              99tests
       A LITTLE WORLD       a little world
A.P Avanashiappa Silk ap avanashiappa silk
             ABC Live             abc live
  ACB (India) Limited    acb india limited
        Accentium Web        accentium web
              Acision              acision
        AddressHealth        addresshealth
           Adhysteria           adhysteria

Dataset 4: 849 → 849 unique


---

## Step 8 — Map Funding Stages to Canonical Taxonomy

Raw stage labels are inconsistent across and within datasets:
- `Series A`, `Series A1`, `Series A+`, `Pre-Series A`, `Pre series A`
- `Seed`, `Seed/ Angel`, `Seed Round`, `Seed Funding`
- `Private Equity Round`, `Private Equity`, `PE`
- `Venture`, `Venture - Series Unknown`, `Venture Round`

We collapse 30+ raw variants into 8 canonical stages:

| Canonical Stage | Maps From |
|---|---|
| Pre-Seed | Pre-seed, pre seed |
| Seed | Seed, Seed Round, Seed Funding, Angel |
| Pre-Series A | Pre-Series A, Pre series A |
| Series A | Series A, Series A1, Series A+ |
| Series B | Series B, Series B1, Series B+ |
| Series C | Series C, Series C1 |
| Late Stage | Series D, E, F, G, H, Pre-IPO |
| PE/Debt | Private Equity, Debt Financing, Post-IPO |

📋 **METHODOLOGY NOTE:** Angel rounds are mapped to Seed rather than
kept separate because angel rounds in India are functionally
equivalent to seed rounds in deal size and timing.

In [12]:
# ─── Build stage taxonomy ─────────────────────────────────

def map_stage(raw_stage):
    if pd.isna(raw_stage):
        return np.nan
    
    s = str(raw_stage).lower().strip()
    
    # Pre-Seed
    if re.search(r'pre[\s-]?seed', s):
        return 'Pre-Seed'
    
    # Seed (+ Angel)
    if re.search(r'^seed|angel', s):
        return 'Seed'
    
    # Pre-Series A
    if re.search(r'pre[\s-]?series?\s*a', s):
        return 'Pre-Series A'
    
    # Series A, B, C (check before Late Stage)
    if re.search(r'series[\s-]?a\b|series a[0-9+]', s):
        return 'Series A'
    if re.search(r'series[\s-]?b\b|series b[0-9+]', s):
        return 'Series B'
    if re.search(r'series[\s-]?c\b|series c[0-9+]', s):
        return 'Series C'
    
    # Late Stage (Series D through H)
    if re.search(r'series[\s-]?[defgh]', s):
        return 'Late Stage'
    
    # PE / Debt / Post-IPO
    if re.search(r'private equity|^pe\b|debt|post[\s-]?ipo', s):
        return 'PE/Debt'
    
    # Venture (Crunchbase catch-all) — keep as unknown Series
    if re.search(r'venture', s):
        return 'Series A'  # default venture to Series A
    
    # Anything else
    return 'Other'

# Apply to Dataset 1
df1['stage_clean'] = df1['stage'].apply(map_stage)

# Show before/after comparison
print("Stage mapping — raw counts vs cleaned counts:")
print("\nRAW stage labels (top 20):")
print(df1['stage'].value_counts().head(20))

print("\n" + "=" * 50)
print("CLEANED canonical stages:")
print(df1['stage_clean'].value_counts())

Stage mapping — raw counts vs cleaned counts:

RAW stage labels (top 20):
stage
Private Equity          1356
Seed Funding            1355
Seed/ Angel Funding       60
Seed / Angel Funding      47
Seed\\nFunding            30
Debt Funding              25
Series A                  24
Seed/Angel Funding        23
Series B                  20
Series C                  14
Series D                  12
Angel / Seed Funding       8
Seed Round                 7
Private Equity Round       4
Seed                       4
Pre-Series A               4
Seed / Angle Funding       3
Series F                   2
Series E                   2
Corporate Round            2
Name: count, dtype: int64

CLEANED canonical stages:
stage_clean
Seed            1542
PE/Debt         1389
Series A          29
Series B          21
Late Stage        18
Other             18
Series C          14
Pre-Series A       9
Name: count, dtype: int64


In [13]:
# ─── Verify stage mapping quality ─────────────────────────

# Check 1 — what raw stages fell into "Other"?
print("What went into 'Other' bucket:")
others = df1[df1['stage_clean'] == 'Other']['stage'].value_counts()
print(others)

# Check 2 — verify PE/Debt bucket looks reasonable
print("\n" + "=" * 50)
print("What went into 'PE/Debt' bucket (top 10):")
pe_debt = df1[df1['stage_clean'] == 'PE/Debt']['stage'].value_counts().head(10)
print(pe_debt)

# Check 3 — \n issue verification
print("\n" + "=" * 50)
print("Rows with embedded newline in raw stage:")
newline_rows = df1[df1['stage'].astype(str).str.contains(r'\\n|\n', na=False, regex=True)]
print(f"Count: {len(newline_rows)}")
print(newline_rows[['stage', 'stage_clean']].head(10))

What went into 'Other' bucket:
stage
Corporate Round         2
Equity                  2
Funding Round           1
Maiden Round            1
Series J                1
Bridge Round            1
Inhouse Funding         1
Mezzanine               1
Equity Based Funding    1
Private Funding         1
Private                 1
Term Loan               1
PrivateEquity           1
Private\\nEquity        1
Crowd funding           1
Crowd Funding           1
Name: count, dtype: int64

What went into 'PE/Debt' bucket (top 10):
stage
Private Equity                 1356
Debt Funding                     25
Private Equity Round              4
Debt and Preference capital       1
Debt                              1
Debt-Funding                      1
Structured Debt                   1
Name: count, dtype: int64

Rows with embedded newline in raw stage:
Count: 31
               stage stage_clean
2933  Seed\\nFunding        Seed
2934  Seed\\nFunding        Seed
2935  Seed\\nFunding        Seed
2936  Seed

In [14]:
# ─── Fix remaining "Other" mappings ──────────────────────

# Manual overrides for specific edge cases
manual_map = {
    'Crowd funding':                'Seed',
    'Crowd Funding':                'Seed',
    'Bridge Round':                 'Pre-Series A',
    'Equity':                       'Series A',
    'Equity Based Funding':         'Series A',
    'Funding Round':                'Series A',
    'Corporate Round':              'PE/Debt',
    'Mezzanine':                    'PE/Debt',
    'Term Loan':                    'PE/Debt',
    'Private Funding':              'PE/Debt',
    'Private':                      'PE/Debt',
    'PrivateEquity':                'PE/Debt',
    'Private\\nEquity':             'PE/Debt',
    'Inhouse Funding':              'Other',
    'Maiden Round':                 'Seed',
    'Series J':                     'Late Stage',
}

# Apply manual overrides only where stage_clean is 'Other'
for raw, canonical in manual_map.items():
    mask = (df1['stage'] == raw) & (df1['stage_clean'] == 'Other')
    df1.loc[mask, 'stage_clean'] = canonical

# Final stage distribution
print("Final canonical stage distribution:")
print(df1['stage_clean'].value_counts())
print(f"\nRemaining 'Other': {(df1['stage_clean'] == 'Other').sum()} rows")

Final canonical stage distribution:
stage_clean
Seed            1545
PE/Debt         1396
Series A          33
Series B          21
Late Stage        19
Series C          14
Pre-Series A      10
Other              2
Name: count, dtype: int64

Remaining 'Other': 2 rows


---

## Step 9 — Consolidate Sectors

Raw industry tags across Dataset 1 have 50+ unique values:
- Inconsistent naming: `E-Tech`, `EdTech`, `Education Technology`
- Over-specific: `Online Pharmacy`, `Telemedicine`, `Health Analytics`
- Misspellings: `Finntech`, `Fintect`

We consolidate to 13 canonical sectors using keyword matching.

📋 **METHODOLOGY NOTE:** Sector mapping uses keyword search on the
raw label, not exact matching. This catches variants and misspellings.
When a label matches multiple sectors, the first match wins —
ordering in the mapping reflects priority (more specific before general).

In [15]:
# ─── Sector consolidation ─────────────────────────────────

# Define canonical sectors with keyword triggers
# Order matters — more specific patterns first
SECTOR_MAP = [
    ('Fintech',      r'fin[\s-]?tech|financial\s*tech|payment|lending|banking|insurance|wealth|neobank|credit|loan'),
    ('Edtech',       r'ed[\s-]?tech|education|e-learning|elearning|learning|tutor|skill|training|school|college'),
    ('Healthtech',   r'health|medical|pharma|clinic|hospital|tele[\s-]?med|wellness|biotech|diagnostics'),
    ('E-commerce',   r'e[\s-]?commerce|marketplace|retail|shopping|d2c|direct.to.consumer|fashion|beauty'),
    ('Foodtech',     r'food|restaurant|delivery|grocery|kitchen|beverage|agri[\s-]?food'),
    ('SaaS/B2B',     r'saas|b2b|enterprise|software|cloud|crm|erp|hr[\s-]?tech|business\s*intel'),
    ('Logistics',    r'logistic|supply\s*chain|shipping|warehouse|freight|delivery\s*tech'),
    ('Mobility',     r'mobility|transport|cab|ride|vehicle|auto|ev|electric\s*vehicle'),
    ('Agritech',     r'agri[\s-]?tech|agriculture|farming|crop|rural'),
    ('Media/Content',r'media|content|entertainment|gaming|sports|music|video|streaming|news'),
    ('Proptech',     r'prop[\s-]?tech|real\s*estate|property|housing|construction'),
    ('Consumer Tech',r'consumer|internet|tech|mobile|app|platform|social'),
    ('Other',        r'.*'),  # catch-all
]

def map_sector(raw_industry):
    if pd.isna(raw_industry):
        return np.nan
    
    s = str(raw_industry).lower().strip()
    
    for canonical, pattern in SECTOR_MAP:
        if re.search(pattern, s):
            return canonical
    
    return 'Other'

# Apply to Dataset 1
df1['sector'] = df1['industry'].apply(map_sector)

# Show results
print("Raw industry (top 20):")
print(df1['industry'].value_counts().head(20))
print("\n" + "=" * 50)
print("Cleaned sector distribution:")
print(df1['sector'].value_counts())

Raw industry (top 20):
industry
Consumer Internet      941
Technology             478
eCommerce              186
Healthcare              70
Finance                 62
ECommerce               61
Logistics               32
E-Commerce              29
Education               24
Food & Beverage         23
Ed-Tech                 14
E-commerce              12
FinTech                  9
Ecommerce                8
IT                       8
Food and Beverage        6
Real Estate              6
Fin-Tech                 6
Others                   6
Health and Wellness      5
Name: count, dtype: int64

Cleaned sector distribution:
sector
Consumer Tech    1632
E-commerce        438
Other             222
Healthtech        127
Foodtech          118
Edtech             88
Logistics          64
Mobility           52
Fintech            48
SaaS/B2B           40
Media/Content      29
Proptech           13
Agritech            2
Name: count, dtype: int64


In [16]:
# ─── Fix over-broad Consumer Tech bucket ─────────────────

# More specific sector map — tighter Consumer Tech definition
SECTOR_MAP_V2 = [
    ('Fintech',       r'fin[\s-]?tech|financial\s*tech|payment|lending|banking|insur|wealth|neobank|credit|loan'),
    ('Edtech',        r'ed[\s-]?tech|education|e-learning|elearning|learning|tutor|skill|training|school'),
    ('Healthtech',    r'health|medical|pharma|clinic|hospital|tele[\s-]?med|wellness|biotech|diagnostic'),
    ('E-commerce',    r'e[\s-]?commerce|marketplace|retail|shopping|d2c|direct.to.consumer|fashion|beauty'),
    ('Foodtech',      r'food|restaurant|grocery|kitchen|beverage|agri[\s-]?food'),
    ('SaaS/B2B',      r'saas|b2b|enterprise|software|cloud|crm|erp|hr[\s-]?tech'),
    ('Logistics',     r'logistic|supply\s*chain|shipping|warehouse|freight'),
    ('Mobility',      r'mobility|transport|cab|ride|vehicle|auto|electric\s*vehicle|\bev\b'),
    ('Agritech',      r'agri[\s-]?tech|agriculture|farming|crop|rural'),
    ('Media/Content', r'media|content|entertainment|gaming|sports|music|video|streaming|news'),
    ('Proptech',      r'prop[\s-]?tech|real\s*estate|property|housing|construction'),
    ('Consumer Tech', r'consumer\s*internet|consumer\s*tech|internet|mobile\s*app|social'),
    ('Technology',    r'technology|tech|software|it\b|digital|platform|app'),
    ('Other',         r'.*'),
]

def map_sector_v2(raw_industry):
    if pd.isna(raw_industry):
        return np.nan
    s = str(raw_industry).lower().strip()
    for canonical, pattern in SECTOR_MAP_V2:
        if re.search(pattern, s):
            return canonical
    return 'Other'

# Apply updated mapping
df1['sector'] = df1['industry'].apply(map_sector_v2)

print("Updated sector distribution:")
print(df1['sector'].value_counts())
print(f"\nTotal mapped: {df1['sector'].notna().sum()}")
print(f"Unmapped (NaN): {df1['sector'].isna().sum()}")

Updated sector distribution:
sector
Consumer Tech    974
Technology       671
E-commerce       437
Other            234
Healthtech       130
Foodtech         105
Edtech            88
Logistics         66
Fintech           48
SaaS/B2B          40
Mobility          36
Media/Content     29
Proptech          13
Agritech           2
Name: count, dtype: int64

Total mapped: 2873
Unmapped (NaN): 171


---

## Step 10 — Reshape Dataset 4: Wide to Long Format

Dataset 4 (Crunchbase) has one row per company with funding amounts
split across 20 columns (seed, round_A, round_B etc.).

We need one row per funding round instead.

**Before (wide):**
| company | seed | round_A | round_B |
|---|---|---|---|
| Flipkart | 500000 | 5000000 | 10000000 |

**After (long):**
| company | stage | amount |
|---|---|---|
| Flipkart | Seed | 500000 |
| Flipkart | Series A | 5000000 |
| Flipkart | Series B | 10000000 |

We use `pd.melt()` for this transformation — one of the most
important reshaping operations in pandas.

📋 **METHODOLOGY NOTE:** Rows where amount = 0 are dropped after
melting — a zero in a round column means that round didn't happen,
not that it raised $0.

In [17]:
# ─── Reshape Dataset 4: wide → long ───────────────────────

# Define the round columns to melt
round_cols = [
    'seed', 'venture', 'angel', 'grant',
    'private_equity', 'round_A', 'round_B',
    'round_C', 'round_D', 'round_E', 'round_F'
]

# Identity columns (stay fixed, don't get melted)
id_cols = [
    'startup_name', 'startup_name_clean', 'industry',
    'market_tag', 'status', 'city', 'founded_year',
    'first_funding_date', 'last_funding_date'
]

# Melt — converts wide to long
df4_long = pd.melt(
    df4,
    id_vars=id_cols,
    value_vars=round_cols,
    var_name='stage_raw',       # column name for the round type
    value_name='amount_usd'     # column name for the amount
)

print(f"Before melt: {df4.shape}")
print(f"After melt:  {df4_long.shape}")
print(f"→ {len(round_cols)} round columns × {len(df4)} companies = {len(round_cols)*len(df4)} rows")

# Drop rows where amount = 0 (round didn't happen)
df4_long = df4_long[df4_long['amount_usd'] > 0].copy()
print(f"After dropping zero-amount rows: {df4_long.shape}")

# Preview
print("\nSample of reshaped data:")
df4_long[['startup_name', 'stage_raw', 'amount_usd', 'city']].head(10)

Before melt: (849, 22)
After melt:  (9339, 11)
→ 11 round columns × 849 companies = 9339 rows
After dropping zero-amount rows: (954, 11)

Sample of reshaped data:


,startup_name,stage_raw,amount_usd,city
4,91Mobiles,seed,"1,000,000.00",Gurgaon
5,99Presents,seed,"20,000.00",Vadodara
6,99tests,seed,"10,000.00",Bangalore
8,A.P Avanashiappa Silk,seed,"75,336.00",Tirupur
14,Adhysteria,seed,"10,000.00",Mumbai
21,Ajahn,seed,"100,000.00",Hyderabad
22,Akosha,seed,"200,000.00",New Delhi
24,AllSchoolStuff.com,seed,"1,500,000.00",Gurgaon
25,AlphaBeta Labs,seed,"40,000.00",Bangalore
42,AppSurfer,seed,"200,000.00",Pune


In [18]:
# ─── Map Dataset 4 stage labels ───────────────────────────

stage_map_d4 = {
    'seed':           'Seed',
    'venture':        'Series A',  # Crunchbase catch-all
    'angel':          'Seed',
    'grant':          'Other',
    'private_equity': 'PE/Debt',
    'round_A':        'Series A',
    'round_B':        'Series B',
    'round_C':        'Series C',
    'round_D':        'Late Stage',
    'round_E':        'Late Stage',
    'round_F':        'Late Stage',
}

df4_long['stage_clean'] = df4_long['stage_raw'].map(stage_map_d4)

print("Dataset 4 stage distribution after mapping:")
print(df4_long['stage_clean'].value_counts())

# Also clean the amount column
# Strip commas from funding_total_usd (Indian format)
df4_long['amount_usd'] = pd.to_numeric(
    df4_long['amount_usd'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

print(f"\nAmount stats for Dataset 4:")
print(f"  Valid amounts: {df4_long['amount_usd'].notna().sum()}")
print(f"  Median: ${df4_long['amount_usd'].median():,.0f}")
print(f"  Max:    ${df4_long['amount_usd'].max():,.0f}")

Dataset 4 stage distribution after mapping:
stage_clean
Series A      490
Seed          267
Series B       92
PE/Debt        37
Series C       37
Late Stage     28
Other           3
Name: count, dtype: int64

Amount stats for Dataset 4:
  Valid amounts: 954
  Median: $4,000,000
  Max:    $2,351,000,000


In [19]:
# ─── Align columns before merge ───────────────────────────

# Tag each dataset with its source
df1['source'] = 'dataset_1'
df4_long['source'] = 'dataset_4'

# Dataset 1 final columns
df1_final = df1[[
    'startup_name', 'startup_name_clean', 'industry', 'sector',
    'city', 'investors', 'stage', 'stage_clean',
    'amount_usd', 'date', 'year', 'quarter', 'month',
    'funding_era', 'source'
]].copy()

# Dataset 4 final columns — match Dataset 1 structure
df4_final = df4_long[[
    'startup_name', 'startup_name_clean', 'industry', 'city',
    'stage_raw', 'stage_clean', 'amount_usd',
    'founded_year', 'status', 'source'
]].copy()

# Add missing columns to Dataset 4 (fill with NaN)
df4_final['sector']      = df4_long['industry'].apply(map_sector_v2)
df4_final['investors']   = np.nan
df4_final['stage']       = df4_long['stage_raw']
df4_final['date']        = np.nan
df4_final['year']        = np.nan
df4_final['quarter']     = np.nan
df4_final['month']       = np.nan
df4_final['funding_era'] = np.nan

# Drop intermediate columns
df4_final.drop(columns=['stage_raw', 'founded_year', 'status'], inplace=True)

print("Dataset 1 final shape:", df1_final.shape)
print("Dataset 4 final shape:", df4_final.shape)
print("\nDataset 1 columns:", list(df1_final.columns))
print("\nDataset 4 columns:", list(df4_final.columns))

Dataset 1 final shape: (3044, 15)
Dataset 4 final shape: (954, 15)

Dataset 1 columns: ['startup_name', 'startup_name_clean', 'industry', 'sector', 'city', 'investors', 'stage', 'stage_clean', 'amount_usd', 'date', 'year', 'quarter', 'month', 'funding_era', 'source']

Dataset 4 columns: ['startup_name', 'startup_name_clean', 'industry', 'city', 'stage_clean', 'amount_usd', 'source', 'sector', 'investors', 'stage', 'date', 'year', 'quarter', 'month', 'funding_era']


In [20]:
# ─── Merge into master dataset ────────────────────────────

master_df = pd.concat([df1_final, df4_final], ignore_index=True)

print(f"Dataset 1 rows:  {len(df1_final)}")
print(f"Dataset 4 rows:  {len(df4_final)}")
print(f"Master total:    {len(master_df)}")
print(f"\nMaster shape: {master_df.shape}")

# Quick sanity check
print("\nSource breakdown:")
print(master_df['source'].value_counts())

print("\nStage distribution in master:")
print(master_df['stage_clean'].value_counts())

print("\nTop 10 sectors:")
print(master_df['sector'].value_counts().head(10))

print("\nMissing values in key columns:")
key_cols = ['startup_name', 'stage_clean', 'sector', 
            'amount_usd', 'city', 'year']
for col in key_cols:
    n = master_df[col].isna().sum()
    pct = n/len(master_df)*100
    print(f"  {col:<20} {n:>4} missing ({pct:.1f}%)")

Dataset 1 rows:  3044
Dataset 4 rows:  954
Master total:    3998

Master shape: (3998, 15)

Source breakdown:
source
dataset_1    3044
dataset_4     954
Name: count, dtype: int64

Stage distribution in master:
stage_clean
Seed            1812
PE/Debt         1433
Series A         523
Series B         113
Series C          51
Late Stage        47
Pre-Series A      10
Other              5
Name: count, dtype: int64

Top 10 sectors:
sector
Consumer Tech    988
Technology       711
E-commerce       610
Other            509
Healthtech       214
SaaS/B2B         181
Edtech           149
Foodtech         108
Media/Content     99
Logistics         72
Name: count, dtype: int64

Missing values in key columns:
  startup_name            0 missing (0.0%)
  stage_clean             4 missing (0.1%)
  sector                201 missing (5.0%)
  amount_usd            979 missing (24.5%)
  city                  183 missing (4.6%)
  year                  962 missing (24.1%)


In [21]:
# ─── Fix year for Dataset 4 rows ──────────────────────────

# Re-attach first_funding_date from df4_long
date_lookup = df4_long[['startup_name_clean', 'first_funding_date']].copy()
date_lookup['first_funding_date'] = pd.to_datetime(
    date_lookup['first_funding_date'], errors='coerce'
)
date_lookup['year_d4'] = date_lookup['first_funding_date'].dt.year
date_lookup = date_lookup.drop_duplicates(subset='startup_name_clean')

# Merge year back into master for Dataset 4 rows
master_df = master_df.merge(
    date_lookup[['startup_name_clean', 'year_d4']],
    on='startup_name_clean',
    how='left'
)

# Fill year where missing using year_d4
master_df['year'] = master_df['year'].fillna(master_df['year_d4'])
master_df.drop(columns=['year_d4'], inplace=True)

# Check improvement
remaining = master_df['year'].isna().sum()
print(f"Year missing before fix: 962")
print(f"Year missing after fix:  {remaining}")
print(f"Years filled in: {962 - remaining}")

print(f"\nYear range in master dataset:")
print(f"  Earliest: {int(master_df['year'].min())}")
print(f"  Latest:   {int(master_df['year'].max())}")

print(f"\nRounds per year:")
print(master_df['year'].value_counts().sort_index())

Year missing before fix: 962
Year missing after fix:  7
Years filled in: 955

Year range in master dataset:
  Earliest: 1999
  Latest:   2020

Rounds per year:
year
1,999.00      1
2,005.00     12
2,006.00     46
2,007.00     67
2,008.00     75
2,009.00     32
2,010.00     70
2,011.00    125
2,012.00    135
2,013.00    201
2,014.00    191
2,015.00    929
2,016.00    993
2,017.00    687
2,018.00    309
2,019.00    111
2,020.00      7
Name: count, dtype: int64


In [22]:
# ─── Final cleanup ────────────────────────────────────────

# Fix year dtype — float → integer
master_df['year'] = master_df['year'].astype('Int64')  # Int64 handles NaN

# Add funding_era for Dataset 4 rows that now have years
master_df['funding_era'] = master_df['year'].apply(
    lambda y: assign_era(y) if pd.notna(y) else np.nan
)

# Standardize city names — common variants
city_map = {
    'Bangalore':  'Bengaluru',
    'Bombay':     'Mumbai',
    'New Delhi':  'Delhi',
    'Delhi':      'Delhi',
    'Gurugram':   'Gurgaon',
    'NCR':        'Delhi',
    'Hyderabad':  'Hyderabad',
}
master_df['city_clean'] = master_df['city'].replace(city_map)

# Reset index
master_df.reset_index(drop=True, inplace=True)

# Final shape and summary
print(f"Master dataset final shape: {master_df.shape}")
print(f"\nFunding era distribution:")
print(master_df['funding_era'].value_counts())
print(f"\nTop 10 cities (cleaned):")
print(master_df['city_clean'].value_counts().head(10))
print(f"\nData types:")
print(master_df.dtypes)

Master dataset final shape: (3998, 16)

Funding era distribution:
funding_era
2014-2016 Consumer Boom      2877
2017-2019 Rationalization    1107
2020-2021 ZIRP Mania            7
Name: count, dtype: int64

Top 10 cities (cleaned):
city_clean
Bengaluru    1106
Mumbai        766
Delhi         562
Gurgaon       423
Chennai       168
Hyderabad     159
Pune          127
Noida         127
Ahmedabad      48
Jaipur         36
Name: count, dtype: int64

Data types:
startup_name              str
startup_name_clean        str
industry                  str
sector                    str
city                      str
investors              object
stage                     str
stage_clean               str
amount_usd            float64
date                   object
year                    Int64
quarter               float64
month                 float64
funding_era               str
source                    str
city_clean                str
dtype: object


In [23]:
# ─── Export cleaned dataset ───────────────────────────────
import sqlite3

# Export 1 — CSV
csv_path = f'{PROCESSED_DIR}funding_clean.csv'
master_df.to_csv(csv_path, index=False)
print(f"✓ CSV exported: {csv_path}")
print(f"  Size: {os.path.getsize(csv_path)/1024:.1f} KB")

# Export 2 — SQLite database
db_path = f'{PROCESSED_DIR}funding.db'
conn = sqlite3.connect(db_path)

master_df.to_sql(
    name='funding_rounds',
    con=conn,
    if_exists='replace',
    index=False
)

# Verify database
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM funding_rounds")
row_count = cursor.fetchone()[0]

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

conn.close()

print(f"\n✓ SQLite database exported: {db_path}")
print(f"  Tables: {[t[0] for t in tables]}")
print(f"  Rows in funding_rounds: {row_count}")
print(f"  Size: {os.path.getsize(db_path)/1024:.1f} KB")

print(f"\n🎉 Clean dataset ready for analysis.")
print(f"   {row_count} funding rounds")
print(f"   {master_df['startup_name'].nunique()} unique companies")
print(f"   {master_df['year'].min()}–{master_df['year'].max()} coverage")

✓ CSV exported: ../data/processed/funding_clean.csv
  Size: 724.0 KB


DatabaseError: Execution failed

In [24]:
# ─── Fix date column then export to SQLite ────────────────

# Convert Timestamp to string for SQLite compatibility
master_df_sql = master_df.copy()
master_df_sql['date'] = master_df_sql['date'].astype(str)
master_df_sql['date'] = master_df_sql['date'].replace('NaT', np.nan)

# Also convert Int64 to regular int64 for SQLite
master_df_sql['year'] = master_df_sql['year'].astype('float64')

# Export to SQLite
db_path = f'{PROCESSED_DIR}funding.db'
conn = sqlite3.connect(db_path)

master_df_sql.to_sql(
    name='funding_rounds',
    con=conn,
    if_exists='replace',
    index=False
)

# Verify
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM funding_rounds")
row_count = cursor.fetchone()[0]
cursor.execute("PRAGMA table_info(funding_rounds)")
columns = cursor.fetchall()
conn.close()

print(f"✓ SQLite exported: {db_path}")
print(f"  Rows: {row_count}")
print(f"  Size: {os.path.getsize(db_path)/1024:.1f} KB")
print(f"\nColumns in database:")
for col in columns:
    print(f"  {col[1]:<25} {col[2]}")

print(f"\n🎉 Clean dataset ready for analysis.")
print(f"   {row_count} funding rounds")
print(f"   {master_df['startup_name'].nunique()} unique companies")

✓ SQLite exported: ../data/processed/funding.db
  Rows: 3998
  Size: 736.0 KB

Columns in database:
  startup_name              TEXT
  startup_name_clean        TEXT
  industry                  TEXT
  sector                    TEXT
  city                      TEXT
  investors                 TEXT
  stage                     TEXT
  stage_clean               TEXT
  amount_usd                REAL
  date                      TEXT
  year                      REAL
  quarter                   REAL
  month                     REAL
  funding_era               TEXT
  source                    TEXT
  city_clean                TEXT

🎉 Clean dataset ready for analysis.
   3998 funding rounds
   3014 unique companies
